# Aula 10 — Checkpoint 4 · Release v2.0 — "O Estoque"
**Computational Thinking with Python · FIAP Rio · Semestre 2**

---

Hoje não tem conteúdo novo: tem **release**. Você junta o que as Aulas 01–09 construíram e entrega a **v2.0** do sistema da sua lanchonete.

- **Rode as células de cima para baixo.** Travou? **Runtime ▸ Restart and run all** (Colab) ou **Run ▸ Run All** (PyCharm/Jupyter).
- Quando aparecer `# TODO`, o código é seu. As células ✅ **CONFIRA** são a mesa de teste da release: todas têm que terminar em *"Tudo certo!"*.


## A release da vez: v2.0 — "O Estoque"

O caixa v1.0 calculava; a v2.0 **controla**. O que ela precisa conter:

1. **Cardápio e estoque em dicionário** — o estoque aninhado `{produto: {"preco": ..., "qtd": ...}}` (Aulas 07–08).
2. **Baixa automática**: vender desconta do estoque na hora, e **ninguém vende o que não tem**.
3. **Filtros por conjunto**: o que está disponível, o que é seguro para um cliente alérgico (Aula 09).
4. **Funções documentadas**: anotação de tipo e docstring em todas — e nenhuma consulta pode levantar `KeyError` (o `get`/`in` é a defesa; `try/except` só chega na Aula 11).

## A rubrica

| critério | o que o professor olha |
|---|---|
| **Roda** | Restart and run all termina sem traceback |
| **Robusto** | produto inexistente, quantidade indisponível: o sistema responde, não cai |
| **Asserts** | todas as células ✅ CONFIRA em "Tudo certo" |
| **PEP 8** | nomes minúsculos com underscore, constantes em MAIÚSCULAS, contratos completos |


## Roteiro de integração

Quatro etapas, cada uma com a sua mesa de teste. As funções das Aulas 07 e 09 entram aqui — pode (e deve) reaproveitar o que você já escreveu; a regra de sempre: **uma função chama a outra, nenhuma conta mora em dois lugares**.

▶️ RODE a célula abaixo: os dados de partida da release.


In [39]:
# --- dados de partida da v2.0 (fornecidos — não altere os valores) ---
LANCHONETE = "Sua Lanchonete"   # 👉 só aqui é seu — batize!
TAXA_ENTREGA = 5.00

ESTOQUE = {
    "X-Burguer": {"preco": 18.50, "qtd": 10},
    "X-Salada": {"preco": 20.00, "qtd": 6},
    "Batata frita": {"preco": 10.00, "qtd": 8},
    "Açaí 300ml": {"preco": 12.00, "qtd": 5},
    "Suco de laranja": {"preco": 8.00, "qtd": 10},
    "Refrigerante lata": {"preco": 6.50, "qtd": 24},
    "Água mineral": {"preco": 4.00, "qtd": 0},
}

INGREDIENTES = {
    "X-Burguer": {"pao", "carne", "queijo"},
    "X-Salada": {"pao", "carne", "queijo", "alface", "tomate"},
    "Batata frita": {"batata", "sal"},
    "Açaí 300ml": {"acai", "granola"},
    "Suco de laranja": {"laranja"},
}

vendas = []   # cada venda registrada: [produto, qtd, valor]

print(f'{LANCHONETE} — montando a release v2.0.')


Sua Lanchonete — montando a release v2.0.


### Etapa 1 — as consultas que não caem (Aulas 07–08)

Duas funções de leitura, nenhuma pode levantar `KeyError`:

- `preco_de(estoque, produto) -> float` — o preço, ou **0.0** se o produto não existe.
- `tem_estoque(estoque, produto, qtd) -> bool` — o produto existe **e** a quantidade basta? (Existir com `qtd` 0 não é ter estoque.)


In [40]:
# TODO: preco_de(estoque, produto) -> float, com contrato — 0.0 para produto inexistente
def preco_de(estoque: dict, produto: str) -> float:
    """Devolve o preço do produto no estoque, ou 0.0 se não existir.

    Usa 'in' antes de acessar, então nunca levanta KeyError.
    """
    if produto in estoque:
        return estoque[produto]['preco']
    return 0.0
# TODO: tem_estoque(estoque, produto, qtd) -> bool, com contrato — cheque com in antes de descer o nível
def tem_estoque(estoque: dict, produto: str, qtd: int) -> bool:
    """Diz se o produto existe e tem quantidade suficiente (qtd pedida).

    Existir com qtd 0 não é ter estoque. Checa com 'in' antes de descer
    ao nível interno, então nunca levanta KeyError.
    """
    if produto in estoque:
        return estoque[produto]['qtd'] >= qtd
    return False


In [41]:
# ✅ CONFIRA — Etapa 1
assert preco_de.__doc__ and tem_estoque.__doc__, "faltou docstring em uma das funções — a rubrica cobra contrato completo"
assert abs(preco_de(ESTOQUE, "X-Burguer") - 18.50) < 0.005, "o X-Burguer custa 18.50"
assert abs(preco_de(ESTOQUE, "Pizza") - 0.0) < 0.005, "produto inexistente devolve 0.0 — sem KeyError"
assert tem_estoque(ESTOQUE, "X-Burguer", 10) == True, "há exatamente 10 X-Burguers — 10 tem que dar True"
assert tem_estoque(ESTOQUE, "X-Burguer", 11) == False, "11 já não tem"
assert tem_estoque(ESTOQUE, "Água mineral", 1) == False, "existir com qtd 0 não é ter estoque"
assert tem_estoque(ESTOQUE, "Pizza", 1) == False, "produto inexistente: False, sem queda"
print("Tudo certo! Etapa 1 fechada — as consultas respondem e nunca caem. ✅")


Tudo certo! Etapa 1 fechada — as consultas respondem e nunca caem. ✅


### Etapa 2 — vender com baixa automática (o coração da release)

`vender(estoque, vendas, produto, qtd) -> float`:

1. **Sem estoque suficiente** (ou produto inexistente): devolve **0.0** e **não altera nada** — use `tem_estoque`.
2. Com estoque: calcula `valor = preco × qtd` (use `preco_de`), **desconta** `qtd` do estoque, registra `[produto, qtd, valor]` em `vendas` e devolve o `valor`.

A docstring diz que a função **altera** o estoque e a lista de vendas recebidos — é decisão de projeto, e declarada.


In [42]:
# TODO: vender(estoque, vendas, produto, qtd) -> float, com contrato completo
# reaproveite tem_estoque e preco_de — nenhuma conta mora em dois lugares
def vender(estoque: dict, vendas: list, produto: str, qtd: int) -> float:
    """Vende qtd do produto: dá baixa no estoque e registra a venda.

    ALTERA o estoque e a lista de vendas recebidos (decisão de projeto).
    Sem estoque suficiente ou produto inexistente: devolve 0.0 e não
    altera nada. Reaproveita tem_estoque e preco_de.
    """
    if not tem_estoque(estoque, produto, qtd):
        return 0.0
    else:
        valor = qtd * preco_de(estoque, produto)
        estoque[produto]['qtd'] -= qtd
        vendas.append([produto, qtd, valor])
        return valor


In [43]:
# ✅ CONFIRA — Etapa 2
estoque_teste = {"X-Burguer": {"preco": 18.50, "qtd": 2}}
vendas_teste = []

v1 = vender(estoque_teste, vendas_teste, "X-Burguer", 2)
assert abs(v1 - 37.00) < 0.005, f"2 × 18.50 = 37.00 — o seu deu {v1:.2f}"
assert estoque_teste["X-Burguer"]["qtd"] == 0, f"a baixa automática falhou: deveriam restar 0, restam {estoque_teste['X-Burguer']['qtd']}"
assert vendas_teste == [["X-Burguer", 2, 37.00]], f"a venda tinha que ser registrada como [produto, qtd, valor] — o seu: {vendas_teste}"

v2 = vender(estoque_teste, vendas_teste, "X-Burguer", 1)
assert abs(v2 - 0.0) < 0.005, "sem estoque: devolve 0.0"
assert len(vendas_teste) == 1, "venda recusada não entra no histórico"
assert vender(estoque_teste, vendas_teste, "Pizza", 1) == 0.0, "produto inexistente: 0.0, sem queda"
print("Tudo certo! Etapa 2 fechada — ninguém vende o que não tem. ✅")


Tudo certo! Etapa 2 fechada — ninguém vende o que não tem. ✅


### Etapa 3 — os filtros por conjunto (Aula 09)

- `disponiveis(estoque) -> set` — os produtos com `qtd > 0` (um `for` sobre `estoque.items()` com `add`).
- `sugestao_segura(ingredientes, estoque, alergias) -> list[str]` — os produtos **com ficha de ingredientes** que estão **disponíveis** e **não cruzam** com as alergias. Lista **já em ordem alfabética**; a docstring avisa que produto sem ficha (bebida industrializada) fica fora da sugestão.


In [44]:
# TODO: disponiveis(estoque) -> set, com contrato

# TODO: sugestao_segura(ingredientes, estoque, alergias) -> list[str], com contrato
# reaproveite disponiveis; a condição é: produto in disponiveis E interseção vazia com alergias
def disponiveis(estoque: dict) -> set:
    """Devolve o conjunto dos produtos com qtd > 0 (em estoque)."""
    conjunto = set()
    for produto, dados in estoque.items():
        if dados['qtd'] > 0:
            conjunto.add(produto)
    return conjunto


def sugestao_segura(ingredientes: dict, estoque: dict, alergias: set) -> list[str]:
    """Produtos com ficha de ingredientes, disponíveis e sem alergias.

    Retorna a lista já em ordem alfabética. Produto sem ficha (bebida
    industrializada) fica de fora da sugestão. Reaproveita disponiveis.
    """
    disp = disponiveis(estoque)
    segura = []
    for produto, ingr_do_produto in ingredientes.items():
        esta_disponivel = produto in disp
        e_seguro = not (ingr_do_produto & alergias)
        if esta_disponivel and e_seguro:
            segura.append(produto)
    return sorted(segura)

In [45]:
# ✅ CONFIRA — Etapa 3
assert disponiveis.__doc__ and sugestao_segura.__doc__, "faltou docstring em uma das funções"
assert disponiveis(ESTOQUE) == {"X-Burguer", "X-Salada", "Batata frita", "Açaí 300ml", "Suco de laranja", "Refrigerante lata"}, "só a Água mineral (qtd 0) fica de fora"
assert sugestao_segura(INGREDIENTES, ESTOQUE, {"tomate"}) == ['Açaí 300ml', 'Batata frita', 'Suco de laranja', 'X-Burguer'], "alérgico a tomate: sai a X-Salada — e a lista vem ordenada"
assert sugestao_segura(INGREDIENTES, ESTOQUE, {"carne"}) == ['Açaí 300ml', 'Batata frita', 'Suco de laranja'], "alérgico a carne: caem os dois lanches"
assert sugestao_segura(INGREDIENTES, ESTOQUE, set()) == ['Açaí 300ml', 'Batata frita', 'Suco de laranja', 'X-Burguer', 'X-Salada'], "sem alergia: todos os disponíveis com ficha"
print("Tudo certo! Etapa 3 fechada — o balcão filtra por conjunto. ✅")


Tudo certo! Etapa 3 fechada — o balcão filtra por conjunto. ✅


### Etapa 4 — fechar o caixa

- `fechar_caixa(vendas) -> float` — o faturamento: a soma da coluna do valor.
- `contagem_por_produto(vendas) -> dict[str, int]` — **unidades vendidas** por produto (o padrão contador da Aula 07/08 — o passo é `+ qtd`, não `+ 1`).


In [46]:
# TODO: fechar_caixa(vendas) -> float, com contrato

# TODO: contagem_por_produto(vendas) -> dict[str, int], com contrato — get com padrão + qtd
def fechar_caixa(vendas: list) -> float:
    """Devolve o faturamento: a soma da coluna do valor de cada venda."""
    total = 0
    for coluna in vendas:
        total += coluna[2]
    return total


def contagem_por_produto(vendas: list) -> dict[str, int]:
    """Conta as unidades vendidas por produto.

    Percorre a lista de vendas (cada item [produto, qtd, valor]) e acumula
    a quantidade por produto. Devolve {produto: total_de_unidades}.
    Usa get com padrão 0 para não levantar KeyError no primeiro registro.
    """
    contagem = {}
    for produto, qtd, valor in vendas:
        contagem[produto] = contagem.get(produto, 0) + qtd
    return contagem

### ✅ A mesa de teste da release

O turno completo, de ponta a ponta: quatro tentativas de venda (uma recusada por falta de estoque), baixa conferida, caixa fechado. Se esta célula terminar em "Release v2.0 conferida", a sua release está de pé.


In [ ]:
# ✅ CONFIRA — a mesa de teste completa da release v2.0
estoque_v2 = {
    "X-Burguer": {"preco": 18.50, "qtd": 10},
    "Refrigerante lata": {"preco": 6.50, "qtd": 24},
    "Água mineral": {"preco": 4.00, "qtd": 0},
}
vendas_v2 = []

vender(estoque_v2, vendas_v2, "X-Burguer", 2)          # 37.00
vender(estoque_v2, vendas_v2, "Refrigerante lata", 3)  # 19.50
vender(estoque_v2, vendas_v2, "Água mineral", 1)       # recusada: qtd 0
vender(estoque_v2, vendas_v2, "X-Burguer", 1)          # 18.50

assert len(vendas_v2) == 3, f"três vendas aceitas e uma recusada — o seu histórico tem {len(vendas_v2)}"
assert abs(fechar_caixa(vendas_v2) - 75.00) < 0.005, f"37.00 + 19.50 + 18.50 = 75.00 — o seu deu {fechar_caixa(vendas_v2):.2f}"
assert estoque_v2["X-Burguer"]["qtd"] == 7, f"10 - 2 - 1 = 7 X-Burguers — o seu estoque diz {estoque_v2['X-Burguer']['qtd']}"
assert estoque_v2["Refrigerante lata"]["qtd"] == 21, "24 - 3 = 21 refrigerantes"
assert contagem_por_produto(vendas_v2) == {"X-Burguer": 3, "Refrigerante lata": 3}, f"unidades por produto: X-Burguer 3, Refrigerante 3 — o seu: {contagem_por_produto(vendas_v2)}"
assert "Água mineral" not in disponiveis(estoque_v2), "a Água mineral segue indisponível"
print("Release v2.0 conferida! Cardápio em dicionário, baixa automática, filtros por conjunto. 🎉 ✅")


Release v2.0 conferida! Cardápio em dicionário, baixa automática, filtros por conjunto. 🎉 ✅


## A defesa oral (sorteio)

No dia do checkpoint, alunos sorteados apresentam a própria release em **3 minutos** e respondem a uma modificação ao vivo. Exemplos do que pode cair:

- *"Onde acontece a baixa de estoque? Mostre a linha."*
- *"Faça o `vender` recusar quantidade zero ou negativa."*
- *"Acrescente o Combo da casa (29.90, 5 unidades) ao estoque e venda 2."*
- *"Por que `preco_de` devolve 0.0 em vez de quebrar? O que muda na Aula 11?"*

Pode usar IA para estudar? Pode. Mas o código é seu: **quem não explica e não altera o próprio código, não fecha o checkpoint.**

## Entrega

- Este notebook completo, com **todas** as células ✅ CONFIRA em "Tudo certo" e o Restart and run all limpo.
- Depois do checkpoint, a **versão de referência da v2.0** fica disponível como código de partida — quem ficou para trás recomeça dela na Aula 11, sem déficit.

**O que vem por aí:** a v2.0 ainda esquece tudo quando o notebook fecha. Na Aula 12 as vendas passam a ser **gravadas em arquivo** — e o sistema começa a ter memória. Antes disso, a Aula 11 traz o `try/except`: a partir de lá, **nenhuma entrada do usuário pode terminar em traceback**.
